In [ ]:
# Importing Necessary Libraries.

import piplite
await piplite.install('seaborn')
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read data set.

df = pd.read_csv('bitcoin.csv')
df.head()

In [ ]:
# Show dataset shape.

df.shape

In [ ]:
# Data discription.

df.describe()

In [ ]:
# Data information.

df.info()

In [ ]:
#Exploratory Data Analysis.

plt.figure(figsize=(15, 5))
plt.plot(df['Close'])
plt.title('Bitcoin Close price.', fontsize=15)
plt.ylabel('Price in dollars.')
plt.show()

In [ ]:
df[df['Close'] == df['Adj Close']].shape, df.shape

In [ ]:
# DataFrame directly in memory.
df = df.drop(['Adj Close'], axis=1)

In [ ]:
# used to count the number of missing (null) values in each column of a DataFrame.

df.isnull().sum()

In [ ]:
features = ['Open', 'High', 'Low', 'Close']

plt.subplots(figsize=(20,10))
for i, col in enumerate(features):
  plt.subplot(2,2,i+1)
  sns.distplot(df[col])
plt.show()

In [ ]:
plt.subplots(figsize=(20,10))
for i, col in enumerate(features):
  plt.subplot(2,2,i+1)
  sns.boxplot(df[col], orient='h')
plt.show()

In [ ]:
# This code is modified by Susobhan Akhuli

splitted = df['Date'].str.split('-', expand=True)

df['year'] = splitted[0].astype('int')
df['month'] = splitted[1].astype('int')
df['day'] = splitted[2].astype('int')

# Convert the 'Date' column to datetime objects
df['Date'] = pd.to_datetime(df['Date']) 

df.head()

In [ ]:
# 'Date' column split: The original 'Date' column was broken down into three new parts.

data_grouped = df.groupby('year').mean()
plt.subplots(figsize=(20,10))
for i, col in enumerate(['Open', 'High', 'Low', 'Close']):
  plt.subplot(2,2,i+1)
  data_grouped[col].plot.bar()
plt.show()

In [ ]:
# Price explosion: Bitcoin prices surged drastically during the year 2021.

df['is_quarter_end'] = np.where(df['month']%3==0,1,0)
df.head()

In [ ]:
df['open-close']  = df['Open'] - df['Close']
df['low-high']  = df['Low'] - df['High']
df['target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)

In [ ]:
# Generate the Python code for the pie chart or help write the model training script.

plt.pie(df['target'].value_counts().values, 
        labels=[0, 1], autopct='%1.1f%%')
plt.show()

In [ ]:
# The heatmap confirms high correlation only among the expected OHLC data, meaning the new features are ready for model building.

plt.figure(figsize=(10, 10))

sns.heatmap(df.corr() > 0.9, annot=True, cbar=False)
plt.show()

In [ ]:
# Features are normalized for faster training and split into a 70/30 ratio to evaluate model performance on unseen data.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Assuming df is already defined
features = df[['open-close', 'low-high', 'is_quarter_end']]
target = df['target']

# Scaling the features
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Split the data into training and validation (test) sets
X_train, X_valid, Y_train, Y_valid = train_test_split(features, target, test_size=0.3, random_state=42)

# 'test_size=0.3' means 30% of the data will be used for testing, and 70% for training

In [ ]:
## Model Development and Evaluation
# We will train Logistic Regression, SVM, and XGBoost models, evaluating them with ROC-AUC scores based on soft probabilities to choose the best performer.
models = [LogisticRegression(), SVC(kernel='poly', probability=True), XGBClassifier()]

for i in range(3):
  models[i].fit(X_train, Y_train)

  print(f'{models[i]} : ')
  print('Training Accuracy : ', metrics.roc_auc_score(Y_train, models[i].predict_proba(X_train)[:,1]))
  print('Validation Accuracy : ', metrics.roc_auc_score(Y_valid, models[i].predict_proba(X_valid)[:,1]))
  print()

In [ ]:
# Model comparison: XGBClassifier performs best but suffers from overfitting, unlike the stable Logistic Regression model.

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(models[0], X_valid, Y_valid, cmap='Blues')
plt.show()